# 08 — Pedestrian Activity

Spatially joins NYC Pedestrian Mobility Plan demand data to census tracts.

**Data source:** NYC DOT Pedestrian Mobility Plan CSV (`needs_pedestrian`).

**Method:** Extract segment midpoints from GeoJSON geometry, match to nearest tract centroid via BallTree, aggregate rank statistics per tract.

**Output columns:** `tract_id`, `pedestrian_rank_mean`, `pedestrian_rank_max`, `pct_high_traffic_segments`

**Output file:** `csv/08_pedestrian_activity.csv`

In [ ]:
ZONES_CONFIG = "zones.json"

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import re
from sklearn.neighbors import BallTree

os.makedirs("csv", exist_ok=True)

with open(ZONES_CONFIG, encoding="utf-8") as f:
    config = json.load(f)

if not config["feature_flags"].get("needs_pedestrian", False):
    print("Pedestrian data not available — skipping notebook 08.")
    df_tracts = pd.read_csv("csv/01_zone_definition.csv", dtype={"tract_id": str})
    df_empty = pd.DataFrame({"tract_id": df_tracts["tract_id"]})
    for col in ["pedestrian_rank_mean", "pedestrian_rank_max", "pct_high_traffic_segments"]:
        df_empty[col] = np.nan
    df_empty.to_csv("csv/08_pedestrian_activity.csv", index=False)
    raise SystemExit("Skipped — needs_pedestrian=false")

PED_PATH = config["pedestrian_path"]
if not os.path.exists(PED_PATH):
    print(f"Pedestrian file not found: {PED_PATH} — skipping.")
    df_tracts = pd.read_csv("csv/01_zone_definition.csv", dtype={"tract_id": str})
    df_empty = pd.DataFrame({"tract_id": df_tracts["tract_id"]})
    for col in ["pedestrian_rank_mean", "pedestrian_rank_max", "pct_high_traffic_segments"]:
        df_empty[col] = np.nan
    df_empty.to_csv("csv/08_pedestrian_activity.csv", index=False)
    raise SystemExit("Skipped — pedestrian file not found")

df_tracts = pd.read_csv("csv/01_zone_definition.csv", dtype={"tract_id": str})
print(f"Loaded {len(df_tracts)} tracts")
print(f"Pedestrian data: {PED_PATH}")

In [ ]:
# ── Load pedestrian data ──────────────────────────────

df_ped = pd.read_csv(PED_PATH)
df_ped["Rank"] = pd.to_numeric(df_ped["Rank"], errors="coerce")
print(f"Pedestrian segments: {len(df_ped):,}")
print(f"Columns: {list(df_ped.columns)}")

# Extract midpoint from the_geom (MULTILINESTRING)
def extract_midpoint(geom_str):
    """Extract approximate centroid from MULTILINESTRING WKT."""
    if pd.isna(geom_str):
        return None, None
    coords = re.findall(r'(-?\d+\.\d+)\s+(-?\d+\.\d+)', str(geom_str))
    if not coords:
        return None, None
    lons = [float(c[0]) for c in coords]
    lats = [float(c[1]) for c in coords]
    return np.mean(lats), np.mean(lons)

print("Extracting segment midpoints...")
midpoints = df_ped["the_geom"].apply(extract_midpoint)
df_ped["seg_lat"] = [m[0] for m in midpoints]
df_ped["seg_lon"] = [m[1] for m in midpoints]

df_ped = df_ped.dropna(subset=["seg_lat", "seg_lon", "Rank"]).copy()
print(f"Segments with valid coordinates: {len(df_ped):,}")

In [ ]:
# ── Match segments to tracts via BallTree ─────────────

tract_coords = np.radians(df_tracts[["tract_lat", "tract_lon"]].values)
tree = BallTree(tract_coords, metric="haversine")

seg_coords = np.radians(df_ped[["seg_lat", "seg_lon"]].values)
distances, indices = tree.query(seg_coords, k=1)

# Earth radius in meters
R = 6371000
df_ped["nearest_tract"] = df_tracts.iloc[indices.flatten()]["tract_id"].values
df_ped["dist_to_tract_m"] = distances.flatten() * R

# Only keep segments within 500m of a tract centroid
df_ped_matched = df_ped[df_ped["dist_to_tract_m"] <= 500].copy()
print(f"Segments matched to tracts (within 500m): {len(df_ped_matched):,}")

In [ ]:
# ── Aggregate per tract ───────────────────────────────

# Define "high traffic" as top 25% rank (lower rank number = higher demand)
rank_threshold = df_ped_matched["Rank"].quantile(0.25)

agg = df_ped_matched.groupby("nearest_tract").agg(
    pedestrian_rank_mean=("Rank", "mean"),
    pedestrian_rank_max=("Rank", "min"),  # min rank = highest demand
    segment_count=("Rank", "count"),
    high_traffic_count=("Rank", lambda x: (x <= rank_threshold).sum()),
).reset_index().rename(columns={"nearest_tract": "tract_id"})

agg["pct_high_traffic_segments"] = (agg["high_traffic_count"] / agg["segment_count"] * 100).round(1)
agg["pedestrian_rank_mean"] = agg["pedestrian_rank_mean"].round(0).astype(int)
agg = agg[["tract_id", "pedestrian_rank_mean", "pedestrian_rank_max", "pct_high_traffic_segments"]]

# Left join to ensure all tracts are present
df_result = df_tracts[["tract_id"]].merge(agg, on="tract_id", how="left")
# Fill missing with worst rank / 0% high traffic
df_result["pedestrian_rank_mean"] = df_result["pedestrian_rank_mean"].fillna(df_ped["Rank"].max()).astype(int)
df_result["pedestrian_rank_max"] = df_result["pedestrian_rank_max"].fillna(df_ped["Rank"].max()).astype(int)
df_result["pct_high_traffic_segments"] = df_result["pct_high_traffic_segments"].fillna(0.0)

print(f"Result: {len(df_result)} tracts")
print(df_result.describe().round(1).to_string())

In [ ]:
# ── Save output ───────────────────────────────────────
output_path = "csv/08_pedestrian_activity.csv"
df_result.to_csv(output_path, index=False, encoding="utf-8")
print(f"Saved: {output_path}  ({len(df_result)} rows x {df_result.shape[1]} cols)")
df_result.head(10)